# Step 1 - 라이브러리 불러오기

In [2]:
import FinanceDataReader as fdr
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import unicodedata
import datetime

print('라이브러리 임포트 완료!') # 잘 불러와졌는지 확인하기 위해 한번 넣어봄

라이브러리 임포트 완료!


# Step 2 - KOSPI 전체 종목 리스트 조회

In [3]:
market = 'KOSPI'
df_market = fdr.StockListing(market)

print(f'전체 종목 수 : {len(df_market)} 개')
print(f'컬럼 목록 : {df_market.columns.tolist()}') # 리스트 형태로 보겠다

df_market.head()

전체 종목 수 : 948 개
컬럼 목록 : ['Code', 'ISU_CD', 'Name', 'Market', 'Dept', 'Close', 'ChangeCode', 'Changes', 'ChagesRatio', 'Open', 'High', 'Low', 'Volume', 'Amount', 'Marcap', 'Stocks', 'MarketId']


,Code,ISU_CD,Name,Market,Dept,Close,ChangeCode,Changes,ChagesRatio,Open,High,Low,Volume,Amount,Marcap,Stocks,MarketId
0,005930,KR7005930003,삼성전자,KOSPI,NaN,281500,1,11000,4.07,269500,288500,262000,28601468,7924104380750,1645727428152000,5846278608,STK
1,000660,KR7000660001,SK하이닉스,KOSPI,NaN,1837000,1,18000,0.99,1780000,1897000,1731000,5346149,9686362948000,1309234244505000,712702365,STK
2,005935,KR7005931001,삼성전자우,KOSPI,NaN,184300,1,4900,2.73,177000,188000,171000,4244408,766267374950,147877012712900,802371203,STK
3,402340,KR7402340004,SK스퀘어,KOSPI,NaN,1080000,2,-18000,-1.64,1087000,1100000,983000,825563,878540383000,142515056880000,131958386,STK
4,005380,KR7005380001,현대차,KOSPI,NaN,663500,2,-36500,-5.21,688000,697000,638000,1928649,1275823435000,135856777741000,204757766,STK


# Step 3 - 한글 종목명 정규화

In [4]:
# 정규화 전후 비교 예시
# s1 = '삼성전자'           # NFC (일반적인 한글 입력)
# s2 = '\u삼\u성\u전\u자'  # NFD (분해된 형태, 눈에는 같아 보임)

def normalize_str(s):
    return unicodedata.normalize('NFKC', s).strip()

# 전각 문자 예시 (실제로 종목명에서 발생할 수 있는 케이스)
test_cases = [
    ('㈜카카오', '정규화 전'),
    (normalize_str('㈜카카오'), '정규화 후'),
]

for text, label in test_cases:
    print(f'{label}: {text}')

# 실제 적용: df_market 종목명 전체 정규화
df_market['Name'] = df_market['Name'].apply(normalize_str)
print('\n✅ 종목명 정규화 완료')
df_market[['Code', 'Name', 'Marcap']].head()

정규화 전: ㈜카카오
정규화 후: (주)카카오

✅ 종목명 정규화 완료


,Code,Name,Marcap
0,005930,삼성전자,1645727428152000
1,000660,SK하이닉스,1309234244505000
2,005935,삼성전자우,147877012712900
3,402340,SK스퀘어,142515056880000
4,005380,현대차,135856777741000


# Step 4 - 시가총액 TOP 10 추출

In [5]:
top10 = df_market.nlargest(10, 'Marcap').iloc[::-1] #반전시키기 위한 ::-1

# 시가총액 단위 변환 : 원 -> 조 (1조 --> 10^12)
top10_display = top10[['Name', 'Marcap']].copy()
top10_display['시가총액(조)'] = (top10_display['Marcap'] / 1e12).round(1)

top10_display[['Name', '시가총액(조)']].reset_index(drop=True)

,Name,시가총액(조)
0,삼성물산,64.5
1,HD현대중공업,65.0
2,두산에너빌리티,71.6
3,삼성전기,77.5
4,LG에너지솔루션,95.0
5,현대차,135.9
6,SK스퀘어,142.5
7,삼성전자우,147.9
8,SK하이닉스,1309.2
9,삼성전자,1645.7


# Step5 - 시가총액 TOP 10 수평 막대그래프

In [6]:
# 그래프 안의 값
fig_top10 = go.Figure(go.Bar(
    x=top10['Marcap'] / 1e12,
    y=top10['Name'],
    orientation = 'h', # 수평 막대그래프(가로)
    text=top10['Marcap'] / 1e12,
    texttemplate = '%{text:.1f}조', # 소수 첫째자리까지, + '조' 단위
    marker_color = 'steelblue' # 막대 색상
))
# 그래프 자체의 레이아웃 설정
fig_top10.update_layout(
    title= f'{market} 시가총액 TOP 10',
    xaxis_title= '시가총액 (조)',
    yaxis_title= '종목명',
    bargap= 0.15, # 막대 사이 간격 (0~1)
    height = 450
)

fig_top10.show()

# Step 6 - 개별 종목 주가 데이터 조회

In [7]:
# 종목명 -> 종목코드 변환
target_name = '삼성전자'
code = df_market.loc[df_market['Name'] == target_name, 'Code'].values[0]
print(f'{target_name} 종목코드: {code}')

삼성전자 종목코드: 005930


In [8]:
# 주가 데이터 조회
start_date = '2024-01-01'
end_date = datetime.datetime.now().strftime('%Y-%m-%d')

df_stock = fdr.DataReader(code, start_date, end_date)

print(f'조회기간: {start_date}~{end_date}')
print(f'데이터 수: {len(df_stock)}일')
df_stock.tail() # 최근 5일

조회기간: 2024-01-01~2026-05-18
데이터 수: 577일


,Open,High,Low,Close,Volume,Change
Date,,,,,,
2026-05-12,290000,291500,266000,279000,41211149,-0.022767
2026-05-13,264000,285500,262000,284000,35540134,0.017921
2026-05-14,282000,299500,282000,296000,39314752,0.042254
2026-05-15,291000,296500,266000,270500,38075487,-0.086149
2026-05-18,269500,288500,262000,281000,33502558,0.038817


# Step 7 - 현재가 / 전일 대비 확인

In [9]:
current = df_stock['Close'].iloc[-1] # 최근 종가
prev = df_stock['Close'].iloc[-2] # 전일 종가
delta = current - prev # 전일 대비 변동폭
pct = delta / prev *100 # 등락률(%)

direction = '🔺' if delta > 0 else '🔻' if delta < 0 else '➖'

print(f'[{target_name}({code})]')
print(f'현재가: {current: ,}원')
print(f'전일 대비: {direction} {abs(delta): ,} 원 ({pct:+.2f}%)')

[삼성전자(005930)]
현재가:  281,000원
전일 대비: 🔺  10,500 원 (+3.88%)


# Step 8 - 종가 라인 차트 (단일 종목)

In [10]:
fig_line = px.line(
    df_stock,
    y='Close',
    title=f'{target_name} 종가 추이',
    labels= {'Close': '종가 (원)', 'Date': '날짜'}
)

fig_line.update_layout(height=400)
fig_line.show()

# Step 9 - 다중 종목 종가 비교

In [11]:
# 비교할 종목 목록
compare_names = ['삼성전자','SK하이닉스', 'LG에너지솔루션']

dfs = []
for name in compare_names:
    matched = df_market.loc[df_market['Name'] == name, 'Code'].values
    if len(matched) == 0:
        print(f'{name}: 종목 코드를 찾을 수 없습니다')
        continue

    c = matched[0]
    df_temp = fdr.DataReader(c, start_date, end_date)

    if not df_temp.empty:
        # Close 열만 추출하고, 열 이름을 종목명으로 변경
        dfs.append(df_temp[['Close']].rename(columns={'Close': name}))
        print(f'{name}({c}) 데이터 로드 완료 ({len(df_temp)}일)')

# 수평 병합
merged_df= pd.concat(dfs, axis=1)
print(f'병합결과: {merged_df.shape}')
merged_df.tail()

삼성전자(005930) 데이터 로드 완료 (577일)
SK하이닉스(000660) 데이터 로드 완료 (577일)
LG에너지솔루션(373220) 데이터 로드 완료 (577일)
병합결과: (577, 3)


,삼성전자,SK하이닉스,LG에너지솔루션
Date,,,
2026-05-12,279000,1835000,443000
2026-05-13,284000,1976000,430000
2026-05-14,296000,1970000,442000
2026-05-15,270500,1819000,417000
2026-05-18,281000,1840000,408000


In [12]:
# 다중 종목 라인 차트
fig_multi = px.line(
    merged_df,
    title='주요 종목 종가 비교',
    labels={'value': '종가 (원)', 'Date': '날짜', 'variable': '종목'}
)

fig_multi.update_layout(height=450, legend_title='종목')
fig_multi.show()

# Step 10 - 캔들 스틱 차트
- go.Candlestick(
        x=날짜,
        open=시가,
        high=고가,
        low=저가,
        close=종가
    ) 

## df_stock.index[-1] --> 가장 최근 날짜
## pd.DateOffset(months=3) --> 달력 기준 3개월
- datetime.timedelta(days)과 비슷하다.
- 월말 처리 부분이 더 정확하다!

In [14]:
# 최근 3개월만 표시
three_months_ago = df_stock.index[-1] - pd.DateOffset(months=3)
# 결과 데이터프레임
df_candle = df_stock[df_stock.index >= three_months_ago] # 최근 3개월 데이터만 True가 나오게끔

In [15]:
# 결과 차트 정보 저장
df_candle

,Open,High,Low,Close,Volume,Change
Date,,,,,,
2026-02-19,188000,190900,187100,190000,27049388,0.048565
2026-02-20,190000,190300,188600,190100,24213880,0.000526
2026-02-23,194400,197600,190500,193000,26364684,0.015255
2026-02-24,193000,200000,192000,200000,28060617,0.036269
2026-02-25,202500,206000,201000,203500,26987996,0.017500
2026-02-26,206500,219000,206000,218000,30095763,0.071253
2026-02-27,210000,223000,210000,216500,51774768,-0.006881
2026-03-03,209500,212500,195100,195100,54879010,-0.098845
2026-03-04,184200,195900,171900,172200,89427954,-0.117376


In [ ]:
df_candle.index # index가 날짜라는 것을 알 수 있다

DatetimeIndex(['2026-02-19', '2026-02-20', '2026-02-23', '2026-02-24',
               '2026-02-25', '2026-02-26', '2026-02-27', '2026-03-03',
               '2026-03-04', '2026-03-05', '2026-03-06', '2026-03-09',
               '2026-03-10', '2026-03-11', '2026-03-12', '2026-03-13',
               '2026-03-16', '2026-03-17', '2026-03-18', '2026-03-19',
               '2026-03-20', '2026-03-23', '2026-03-24', '2026-03-25',
               '2026-03-26', '2026-03-27', '2026-03-30', '2026-03-31',
               '2026-04-01', '2026-04-02', '2026-04-03', '2026-04-06',
               '2026-04-07', '2026-04-08', '2026-04-09', '2026-04-10',
               '2026-04-13', '2026-04-14', '2026-04-15', '2026-04-16',
               '2026-04-17', '2026-04-20', '2026-04-21', '2026-04-22',
               '2026-04-23', '2026-04-24', '2026-04-27', '2026-04-28',
               '2026-04-29', '2026-04-30', '2026-05-04', '2026-05-06',
               '2026-05-07', '2026-05-08', '2026-05-11', '2026-05-12',
      

In [19]:
# 캔들 차트
fig_candle = go.Figure(data=[go.Candlestick(
    x= df_candle.index, # 날짜
    open=df_candle['Open'],
    high=df_candle['High'],
    low=df_candle['Low'],
    close=df_candle['Close']
)])

# 캔들 차트 - 디자인
fig_candle.update_layout(
    title=f'{target_name} 캔들 스틱 차트(최근 3개월)',
    xaxis_title='날짜',
    yaxis_title='가격(원)',
    xaxis_rangeslider_visible=False, # 하단 범위 슬라이더 숨김, 
    height=500
)

fig_candle.show()